In [ ]:
# v6: ResNet1DGRU + EEGDatasetV2 (mu-law) + eeg_id aggregation + GroupKFold
import os, sys, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupKFold
from tqdm.notebook import tqdm

In [ ]:
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    DATA_ROOT   = '/kaggle/input/competitions/hms-harmful-brain-activity-classification'
    CODE_DIR    = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code'
    META_PATH   = os.path.join(DATA_ROOT, 'train.csv')
else:
    DATA_ROOT   = os.path.abspath('../')
    CODE_DIR    = os.path.abspath('../project')
    META_PATH   = os.path.abspath('../data_raw/train.csv')

PARQUET_DIR = os.path.join(DATA_ROOT, 'train_eegs')

sys.path.insert(0, CODE_DIR)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Local'} | Device: {DEVICE}")
print(f"META_PATH   : {META_PATH}")
print(f"PARQUET_DIR : {PARQUET_DIR}")
print(f"CODE_DIR    : {CODE_DIR}")

In [ ]:
cfg = {
    "seed": 42,
    "model": {
        "name"       : "eeg_1d_resnet_gru",
        "num_classes": 6,
        "in_channels": 8,
        "feat_dim"   : 256,
        "input_T"    : 2000,
    },
    "train": {
        "epochs"              : 30,
        "batch_size"          : 64,
        "lr"                  : 1e-3,
        "weight_decay"        : 1e-2,
        "amp"                 : True,
        "grad_clip"           : 1.0,
        "early_stop_patience" : 10,
        "num_workers"         : 4,
        "val_fold"            : 0,
        "n_splits"            : 5,
    },
}
print(cfg)

In [ ]:
from src.data import EEGDatasetV2

TARGETS    = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
EXPERT_TO_VOTE = {
    'Seizure': 'seizure_vote', 'LPD': 'lpd_vote', 'GPD': 'gpd_vote',
    'LRDA': 'lrda_vote', 'GRDA': 'grda_vote', 'Other': 'other_vote',
}
EXPERT_TO_IDX = {'Seizure': 0, 'LPD': 1, 'GPD': 2, 'LRDA': 3, 'GRDA': 4, 'Other': 5}

# 1. Load raw train.csv
raw = pd.read_csv(META_PATH)

# 2. Group by eeg_id — sum votes, keep first patient_id and expert_consensus
agg = raw.groupby('eeg_id', as_index=False).agg(
    {**{t: 'sum' for t in TARGETS},
     'patient_id'      : 'first',
     'expert_consensus': 'first'}
)

# 3. Add +10 to expert_consensus vote before normalising
for idx, row in agg.iterrows():
    ec = row['expert_consensus']
    if ec in EXPERT_TO_VOTE:
        agg.at[idx, EXPERT_TO_VOTE[ec]] += 10

# 4. Normalise votes to sum=1 per row
vote_sums = agg[TARGETS].sum(axis=1)
agg[TARGETS] = agg[TARGETS].div(vote_sums, axis=0)

# 5. Store soft_y as numpy array of 6 floats
agg['soft_y'] = list(agg[TARGETS].to_numpy(dtype=np.float32))

# 6. Map expert_consensus to int target
agg['target'] = agg['expert_consensus'].map(EXPERT_TO_IDX).astype(int)

print(f"Unique EEGs after aggregation: {len(agg):,}")

# 7–8. GroupKFold by patient_id; val_fold=0
gkf    = GroupKFold(n_splits=cfg["train"]["n_splits"])
splits = list(gkf.split(agg, groups=agg['patient_id'].values))
train_idx, val_idx = splits[cfg["train"]["val_fold"]]
train_df = agg.iloc[train_idx].reset_index(drop=True)
val_df   = agg.iloc[val_idx  ].reset_index(drop=True)
print(f"Train EEGs: {len(train_df):,} | Val EEGs: {len(val_df):,}")

# 9. Build datasets
train_ds = EEGDatasetV2(train_df, parquet_dir=PARQUET_DIR)
val_ds   = EEGDatasetV2(val_df,   parquet_dir=PARQUET_DIR)

# Collate with 5× downsampling: T 10000 → 2000
def collate_fn(batch):
    xs    = torch.stack([torch.tensor(b["x"][:, ::5], dtype=torch.float32) for b in batch])
    ys    = torch.tensor([b["y"] for b in batch], dtype=torch.long)
    softs = torch.stack([b["soft_y"] for b in batch])
    return {"x": xs, "y": ys, "soft_y": softs}

# 10. Build DataLoaders
train_loader = DataLoader(train_ds, batch_size=cfg["train"]["batch_size"],
                          shuffle=True,  num_workers=cfg["train"]["num_workers"],
                          pin_memory=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=cfg["train"]["batch_size"],
                          shuffle=False, num_workers=cfg["train"]["num_workers"],
                          pin_memory=True, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# Sanity-check batch shapes
batch = next(iter(train_loader))
print(f"x shape  : {batch['x'].shape}")
print(f"soft_y   : {batch['soft_y'].shape}")
print(f"target   : {batch['y'][:4]}")

In [ ]:
from src.models.classifier import build_model

model = build_model(cfg["model"]).to(DEVICE)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters — total: {total_params:,} | trainable: {trainable_params:,}")

In [ ]:
NUM_EPOCHS = cfg["train"]["epochs"]
LR         = cfg["train"]["lr"]
GRAD_CLIP  = cfg["train"]["grad_clip"]
USE_AMP    = cfg["train"]["amp"] and DEVICE.type == "cuda"
patience   = cfg["train"]["early_stop_patience"]

criterion = nn.KLDivLoss(reduction='batchmean')
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=cfg["train"]["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

best_kl, history = float('inf'), []
wait = 0

for epoch in range(1, NUM_EPOCHS + 1):
    # ── train ────────────────────────────────────────────────────────────────
    model.train()
    t0         = time.time()
    train_loss = 0.0
    for batch in train_loader:
        x      = batch["x"].to(DEVICE)
        soft_y = batch["soft_y"].to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model(x)
            loss   = criterion(F.log_softmax(logits, dim=1), soft_y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    scheduler.step()

    # ── validate ─────────────────────────────────────────────────────────────
    model.eval()
    val_kl_total = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            x      = batch["x"].to(DEVICE)
            soft_np = batch["soft_y"].numpy()
            y       = batch["y"]
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                logits = model(x)
            prob = F.softmax(logits, dim=1).cpu().numpy()
            kl   = (np.clip(soft_np, 1e-7, 1) *
                    np.log(np.clip(soft_np, 1e-7, 1) / np.clip(prob, 1e-7, 1))).sum(axis=1).mean()
            val_kl_total += kl
            all_preds .extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(y.tolist())

    avg_train = train_loss   / len(train_loader)
    val_kl    = val_kl_total / len(val_loader)
    macro_f1  = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    elapsed   = time.time() - t0

    history.append({"epoch": epoch, "train_kl": avg_train,
                    "val_kl": val_kl, "macro_f1": macro_f1})
    print(f"Epoch {epoch:03d} | train_kl {avg_train:.4f} | "
          f"val_kl {val_kl:.4f} | macro_f1 {macro_f1:.4f} | {elapsed:.0f}s")

    if val_kl < best_kl:
        best_kl = val_kl
        wait    = 0
        torch.save(model.state_dict(), "best_model_v6.pt")
        print(f"  ✓ saved best model (val_kl={best_kl:.4f})")
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nTraining complete. Best val KL: {best_kl:.4f}")

In [ ]:
epochs   = [h["epoch"]    for h in history]
train_kl = [h["train_kl"] for h in history]
val_kl   = [h["val_kl"]   for h in history]
macro_f1 = [h["macro_f1"] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(epochs, train_kl, label="train KL")
ax1.plot(epochs, val_kl,   label="val KL")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("KL Divergence")
ax1.set_title("KL Divergence (train vs val)"); ax1.legend()

ax2.plot(epochs, macro_f1, color="green")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Macro F1")
ax2.set_title("Validation Macro F1")

plt.tight_layout()
plt.savefig("training_curves_v6.png", dpi=150)
plt.show()

best_kl_epoch  = epochs[val_kl.index(min(val_kl))]
best_f1_epoch  = epochs[macro_f1.index(max(macro_f1))]
print(f"\nFinal epoch summary:")
print(f"  Best val KL  : {min(val_kl):.4f}  (epoch {best_kl_epoch})")
print(f"  Best macro F1: {max(macro_f1):.4f}  (epoch {best_f1_epoch})")
print(f"  Final train KL: {train_kl[-1]:.4f}")
print(f"  Final val KL  : {val_kl[-1]:.4f}")